# Serve a model with Ollama

Starts Ollama on this job's GPU, makes sure the model is downloaded and loaded, and publishes the endpoint so the laptop can tunnel to it.

- **Interactive:** open this notebook in a `mynotebook` GPU session and *Run All*. The server lives as long as the kernel; run the last cell to stop it.
- **Headless:** `model/deploy/up.sh` runs the same cells as a Slurm job and keeps serving until the job's walltime.

Either way, `model/deploy/connect.sh` on the laptop opens the tunnel.

In [ ]:
import os
import subprocess

from ollama_server import OllamaServer, load_config

config = load_config()
MODEL = os.environ.get("SERVE_MODEL") or config["DEFAULT_MODEL"]
print("model:", MODEL)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU visible")

In [ ]:
server = OllamaServer(config)
server.start()
server.pull(MODEL)
print(f"loaded {MODEL} in {server.warm(MODEL):.1f}s")
for m in server.loaded():
    print(f"  {m['name']}: {m['size_vram'] / 2**30:.1f} GiB in VRAM")

In [ ]:
print("smoke test:", repr(server.generate(MODEL, "Reply with the single word: ready").strip()))
print("endpoint file:", server.publish(MODEL))
print(server.connection_help())

In [ ]:
if os.environ.get("SERVE_HEADLESS") == "1":
    server.serve_forever()

Interactive sessions only: stop the server and remove the endpoint file.

In [ ]:
server.stop()